In [6]:
import requests
import datetime

from utils.ingestion import Ingestion 
from utils.file_utils import FileUtils

In [7]:
ingestion = Ingestion()

configs = ingestion.load_config_json("cities", "metrics")

today = datetime.date.today().isoformat()
today_str = today.replace("-", "")
print(f"Current date: {today}")

Current date: 2026-04-23


In [ ]:
base_url = "https://api.open-meteo.com/v1/forecast"

metrics = configs['metrics']
cities = configs['cities']

hourly_metrics = metrics.get("hourly")

for city, coordinates in cities.items():
    latitude = coordinates["latitude"]
    longitude = coordinates["longitude"]

    params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": ",".join(hourly_metrics),
    "forecast_days": 7,
    "timezone": "UTC"
}
    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        print(f"\n {city}")
        print(data["hourly"]["apparent_temperature"][:5])  
    else:
        print(f"Error getting data for {city}")
    
    try:
        FileUtils.write_json(data, f"../data/bronze/open_meteo/forecast/{today_str}/{city}_{today_str}.json")
    except Exception as e:
        print(f"Error writing JSON file for {city}: {e}")